In [0]:

from datetime import datetime

modo = "historico" # "automatico"


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None

  

In [0]:
if modo == "automatico":
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW tvw_fact_cartera AS
        SELECT
     o.id_oblig                                                      AS ID_OBLIGACION
    ,o.id_cli                                                        AS ID_CLIENTE
    ,o.cod_prod                                                      AS CODIGO_PRODUCTO
    ,o.ciudad                                                        AS CIUDAD
    ,o.pais                                                          AS PAIS

    -- Saldos
    ,o.vr_aprobado                                                   AS VALOR_APROBADO
    ,o.vr_desembolsado                                               AS VALOR_DESEMBOLSADO
    ,o.sdo_capital                                                   AS SALDO_CAPITAL
    ,o.sdo_interes                                                   AS SALDO_INTERES
    ,o.sdo_capital + o.sdo_interes                                   AS SALDO_TOTAL

    -- Fechas
    ,o.fec_desembolso                                                AS FECHA_DESEMBOLSO
    ,o.fec_venc                                                      AS FECHA_VENCIMIENTO
    ,o.num_cuotas_pend                                               AS CUOTAS_PENDIENTES

    -- Mora
    ,o.dias_mora_act                                                 AS DIAS_MORA

    -- Bucket de mora (5 rangos)
    ,CASE
        WHEN o.dias_mora_act = 0                THEN 'AL DIA'
        WHEN o.dias_mora_act BETWEEN 1  AND 30  THEN 'RANGO 1'
        WHEN o.dias_mora_act BETWEEN 31 AND 60  THEN 'RANGO 2'
        WHEN o.dias_mora_act BETWEEN 61 AND 90  THEN 'RANGO 3'
        WHEN o.dias_mora_act > 90               THEN 'DETERIORADO'
        ELSE 'SIN_CLASIFICAR'
     END                                                             AS BUCKET_MORA

    -- Clasificacion regulatoria (Superfinanciera Colombia)
    ,CASE
        WHEN o.dias_mora_act = 0               THEN 'A'
        WHEN o.dias_mora_act BETWEEN 1 AND 30  THEN 'B'
        WHEN o.dias_mora_act BETWEEN 31 AND 60 THEN 'C'
        WHEN o.dias_mora_act BETWEEN 61 AND 90 THEN 'D'
        WHEN o.dias_mora_act > 90              THEN 'E'
        ELSE 'A'
     END                                                             AS CLASIFICACION_REGULATORIA

    -- Porcentaje de provision por categoria (tabla regulatoria Superfinanciera)
    ,CASE
        WHEN o.dias_mora_act = 0               THEN 0.01
        WHEN o.dias_mora_act BETWEEN 1 AND 30  THEN 0.035
        WHEN o.dias_mora_act BETWEEN 31 AND 60 THEN 0.20
        WHEN o.dias_mora_act BETWEEN 61 AND 90 THEN 0.50
        WHEN o.dias_mora_act > 90              THEN 1.00
        ELSE 0.01
     END                                                             AS PCT_PROVISION

    -- Provision estimada sobre saldo capital
    ,ROUND(o.sdo_capital *
        CASE
            WHEN o.dias_mora_act = 0               THEN 0.01
            WHEN o.dias_mora_act BETWEEN 1 AND 30  THEN 0.035
            WHEN o.dias_mora_act BETWEEN 31 AND 60 THEN 0.20
            WHEN o.dias_mora_act BETWEEN 61 AND 90 THEN 0.50
            WHEN o.dias_mora_act > 90              THEN 1.00
            ELSE 0.01
        END
    , 2)                                                             AS PROVISION_ESTIMADA

    ,o.calif_riesgo                                                  AS CALIFICACION_RIESGO_ORIGEN
    ,o.periodo                                                       AS PERIODO
    ,CURRENT_DATE                                                    AS _FECHA_CARGA
    ,'silver.cleaned.tb_obligaciones'                                AS _FUENTE

FROM silver.cleaned.tb_obligaciones o
INNER JOIN gold.financiero.dim_clientes c
    ON o.id_cli = c.ID_CLIENTE
WHERE o.periodo = '{periodo}'
    """)

    spark.sql(f"""
        DELETE FROM gold.financiero.fact_cartera
        WHERE PERIODO = '{periodo}'
    """)

    spark.sql("""
        INSERT INTO gold.financiero.fact_cartera
        SELECT
            ID_OBLIGACION                
            ,ID_CLIENTE                   
            ,CODIGO_PRODUCTO        
            ,CIUDAD              
            ,PAIS                  
            ,VALOR_APROBADO      
            ,VALOR_DESEMBOLSADO      
            ,SALDO_CAPITAL           
            ,SALDO_INTERES              
            ,SALDO_TOTAL              
            ,FECHA_DESEMBOLSO        
            ,FECHA_VENCIMIENTO      
            ,CUOTAS_PENDIENTES        
            ,DIAS_MORA                
            ,BUCKET_MORA                 
            ,CLASIFICACION_REGULATORIA    
            ,PCT_PROVISION               
            ,PROVISION_ESTIMADA           
            ,CALIFICACION_RIESGO_ORIGEN   
            ,PERIODO                      
            ,_FECHA_CARGA                 
            ,_FUENTE                      
        FROM tvw_fact_cartera
    """)

    spark.sql("""
        INSERT INTO gold.financiero.fact_cartera
        SELECT
            m.ID_MOVIMIENTO                                                  AS ID_MOVIMIENTO
        FROM tvw_fact_cartera
    """)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.financiero.fact_cartera (
     ID_OBLIGACION                STRING
     ,ID_CLIENTE                   STRING
     ,CODIGO_PRODUCTO              STRING
     ,CIUDAD                       STRING
     ,PAIS                         STRING
     ,VALOR_APROBADO               DOUBLE
     ,VALOR_DESEMBOLSADO           DOUBLE
     ,SALDO_CAPITAL                DOUBLE
     ,SALDO_INTERES                DOUBLE
     ,SALDO_TOTAL                  DOUBLE
     ,FECHA_DESEMBOLSO             DATE
     ,FECHA_VENCIMIENTO            DATE
     ,CUOTAS_PENDIENTES            LONG
     ,DIAS_MORA                    LONG
     ,BUCKET_MORA                  STRING
     ,CLASIFICACION_REGULATORIA    STRING
     ,PCT_PROVISION                DOUBLE
     ,PROVISION_ESTIMADA           DOUBLE
     ,CALIFICACION_RIESGO_ORIGEN   STRING
     ,PERIODO                      STRING
     ,_FECHA_CARGA                 DATE
     ,_FUENTE                      STRING
)
USING DELTA
PARTITIONED BY (PERIODO)
LOCATION 'abfss://gold@stdataknowdeveastus001.dfs.core.windows.net/financiero/fact_cartera';